# Chain Techniques [Langchain]:

In [1]:
import os 
from dotenv import load_dotenv
# 
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
load_dotenv()

True

### Initializing the LLM:

In [3]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature = 1,
    max_tokens = None, 
    max_retries = 2
)

## Chains:

In [4]:
from langchain.chains import LLMChain, SimpleSequentialChain, SequentialChain
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE

### Type 1: Simple and basic chain:

In [5]:
prompt = ChatPromptTemplate.from_template(
    "Give me a short, clever and relevant information about {topic}."
)

In [6]:
output_parser = StrOutputParser()

In [7]:
chain = prompt | llm | output_parser

In [8]:
response = chain.invoke(
    {"topic": "Artificial Intelligence"}
)
print(response)

Here are a few options:

1.  **AI: Humanity's mirror, reflecting our data back as insight.**
2.  **AI: The silicon brain that learns faster than you can think.**
3.  **AI: From 'if-then' statements to 'what if' possibilities.**


### Type 2: Sequential Chain:

In [23]:
# Chain 1: Name Generator for Brand
prompt_1 = ChatPromptTemplate.from_template(
    "What is a creative, unique and out-of-the box name for a company that sells {product}? [List out only 5 possible names no other messages or output spamming. Do as asked strictly]"
)

chain_1 = prompt_1 | llm | StrOutputParser()


# Chain 2: Brand's Tagline Generator based on brand name.
prompt_2 = ChatPromptTemplate.from_template(
    "Write a catchy, unique and best tagline for the company: {company_name}. [List out only 5 related taglines no other messages or output spamming. Do as asked strictly. Give output in JSON format]"
)

chain_2 = prompt_2 | llm | StrOutputParser()

In [24]:
# Combining both the above chains sequentially...

seq_chain = chain_1 | (lambda name: {"company_name" : name}) | chain_2

In [25]:
response = seq_chain.invoke({
    "product" : "Laptops & Computer Hardware"
})
print(response)

```json
[
  {
    "company": "VoxelWorks",
    "tagline": "Building Tomorrow, Block by Block."
  },
  {
    "company": "Synapse Systems",
    "tagline": "Connecting Minds, Empowering Futures."
  },
  {
    "company": "LumenCore",
    "tagline": "Illuminating Innovation's Core."
  },
  {
    "company": "Epoch Compute",
    "tagline": "Defining the Next Era of Intelligence."
  },
  {
    "company": "Axiom Gear",
    "tagline": "Engineering Principles, Empowering Performance."
  }
]
```


---

### Type 3: Advanced Sequential Chain (with multiple I/O):

- I will try to build a chain structure that is responsible for generating the movie title based on genre, suitable casts.

In [27]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableBranch

In [ ]:
# Chain 1:::::
title_prompt = ChatPromptTemplate.from_template(
    "Give me a compelling title for a {genre} based movie in the {era}. [List out only one title no other messages or output spamming. Do as asked strictly]"
)
chain_title = title_prompt | llm | StrOutputParser()

# Chain 2:::::
suitable_casts_prompt = ChatPromptTemplate.from_template(
    "Give me two suitable Casts (actors) Name for the given {movie_title}. [List out only single related casts/actors [actor & actress name] no other messages or output spamming. Do as asked strictly]."
)
chain_casts = suitable_casts_prompt | llm | StrOutputParser()


# Final Chain::::
movie_ideation_chain = RunnablePassthrough.assign(
    movie_title = chain_title
    ) | RunnablePassthrough.assign(
        casts = chain_casts
    )

In [40]:
response = movie_ideation_chain.invoke(
    {   
        "genre" : "horror",
        "era" : "Bollywood"
    }
)
response

{'genre': 'horror',
 'era': 'Bollywood',
 'movie_title': 'Aahat',
 'casts': 'Nawazuddin Siddiqui & Tripti Dimri\nJaideep Ahlawat & Radhika Apte'}

### Type 4: Router Chain:

- via LCEL - LangChain Expression Language.
- `RunnableBranch` - Conditional based branch selection.

Here, I am gonna build two kind of chains that are to run based on the inputs (conditional branching)

In [41]:
# CHAIN 1 | History Specialist Chain :::::

prompt_hist = ChatPromptTemplate.from_template(
    """You are an excellent History Professor.
    Answer the following Question simply, and briefly in simple text format and single small paragraph [You are restricted to answer any unrelated question]:
    
    Question : {input}
    """
)
hist_chain = prompt_hist | llm | StrOutputParser()  

# CHAIN 2 | Computer Science Specialist Chain :::::
prompt_compsci = ChatPromptTemplate.from_template(
    """You are a Computer Science Expert Engineer & professor.
    Answer the following Question simply, and briefly in simple text format and single small paragraph [You are restricted to answer any unrelated question]:
    
    Question: {input}
    """
)
compsci_chain = prompt_compsci | llm | StrOutputParser()


# GENERAL CHAIN | General Question answering chain :::::
prompt_general = ChatPromptTemplate.from_template(
    """Answer the following question briefly and in single paragraph: {input}
    """
)
general_chain = prompt_general | llm | StrOutputParser()

In [ ]:
# ROUTING LOGIC DEFINITION:::::
prompt_routing = ChatPromptTemplate.from_template(
    """Given the user question below, classify it as either: 'history', 'computer_science', or 'general'.
    
    Do not respond with more than one word.
    
    <question>
    {input}
    </question>
    
    Classification:"""
)
router_chain = prompt_routing | llm | StrOutputParser()

In [43]:
# Runnable Branch [Conditional Branch selection]:::::
branch = RunnableBranch(
    (lambda x: "history" in x['topic'].lower(), hist_chain),
    (lambda x: "computer_science" in x["topic"].lower(), compsci_chain),
    general_chain,
)

In [44]:
# FULL & FINAL Chain:::::
final_router_chain = {
    "topic" : router_chain,
    "input" : lambda x: x["input"]} | branch

In [45]:
# TEST:::::

compsci_res = final_router_chain.invoke(
    {"input" : "what is meant by AI Agents?"}
)
hist_res = final_router_chain.invoke(
    {"input" : "What is Indus Valley civilization?"}
)
general_res = final_router_chain.invoke(
    {"input" : "What are the best practices to stay healthy and fit?"}
)


In [46]:
print(f"""
COMPUTER SCIENCE ques:
{compsci_res}
{'-'*75}
History ques:
{hist_res}
{'-'*75}
Genral ques:
{general_res}
{'-'*75}""")


COMPUTER SCIENCE ques:
An AI agent is an autonomous software entity designed to perceive its environment, process that information, and then take actions to achieve specific goals or objectives. Operating often continuously and adaptively, it acts on behalf of a user or system, leveraging its internal reasoning and decision-making capabilities to respond to changing conditions and fulfill its assigned tasks, much like a digital assistant or a component within a larger system.
---------------------------------------------------------------------------
History ques:
The Indus Valley Civilization, also known as the Harappan Civilization, was an ancient Bronze Age civilization located in what is now Pakistan and northwest India. Flourishing from roughly 2500 to 1900 BCE, it was one of the world's earliest major urban cultures, notable for its advanced city planning, sophisticated water management systems, and a unique, still-undeciphered writing system.
-----------------------------------

---
By Kirtan Ghelani `@SculptSoft`